# Lab 5 

This week we'll discuss how to solve for the path of $\left\{k_t,c_t\right\}$ using Euler equations. In the notes here, I am using $k_{t-1}$ to denote start of period capital and $k_t$ to denote end of period capital.

We'll consider the basic model with non-valued leisure. The agent's flow utility function is $u\left(c\right)$. The agent has a production technology $F\left( \cdot \right)$ that converts $k$ units of capital into $F\left(k\right)$ units of output, which can be converted into consumption or end of period capital.

The Euler equation for capital is 
\begin{align}
    u^{\prime}\left(c_{t}\right)=\beta u^{\prime}\left(c_{t+1}\right)\left[F^{\prime}\left(k_t\right)\right]
\end{align}

In our specific example, we will let 
\begin{align}
    u\left(c_t\right)\triangleq \frac{c_t^{1-\gamma}-1}{1-\gamma} \\
    F\left(k_t\right) \triangleq k_t^{\alpha}+\left(1-\delta\right)k_t
\end{align}

Since all output is used for consumption or new capital, we have 
\begin{align}
    c_t+k_t=k_{t-1}^{\alpha}+\left(1-\delta\right)k_{t-1}
\end{align}

Substituting out for $c_t$ and $c_{t+1}$ in the Euler equation for capital we have
\begin{align}
    \left(k_{t-1}^{\alpha}+\left(1-\delta\right)k_{t-1}-k_t\right)^{-\gamma}=&\\
    & \beta \left(k_{t}^{\alpha}+\left(1-\delta\right)k_{t}-k_{t+1}\right)^{-\gamma}\left(\alpha k_t^{\alpha-1}+\left(1-\delta\right)\right) 
\end{align}

I have one of these conditions for each period $t\in \mathbb{N}$. Since $k_{-1}$ is given, the solution to the model is vector $K \in \mathbb{R}^{\infty}$ such that $H\left(K\right)=0$ where $H\left(X\right)$ is 

\begin{align}
H\left(X\right)=
\begin{bmatrix}
h\left(x_{-1},x_0,x_1\right) \\
    h\left(x_0,x_1,x_2\right) \\
    \vdots \\
    h\left(x_{j-2},x_{j-1},x_j\right) \\
    \vdots
\end{bmatrix}
\end{align}

and 

\begin{align}
h\left(x_{j-1},x_{j},x_{j+1}\right) \triangleq \left(x_{t-1}^{\alpha}+\left(1-\delta\right)x_{t-1}-x_t\right)^{-\gamma}-
    & \beta \left(x_{t}^{\alpha}+\left(1-\delta\right)x_{t}-x_{t+1}\right)^{-\gamma}\left(\alpha x_t^{\alpha-1}+\left(1-\delta\right)\right)  
\end{align}

Without further information, solving this problem looks hopeless. We need to solve and ininite number of equations for an infinite number of unknowns. The way to reduce this to a finite dimensional system is to use properties of the solution. Specifically, we know there exits a steady state. Therefore, $\forall \tau> T$, $\left|k_{\tau}-k^*\right| <\epsilon$. Therefore, instead of solving $H\left(X\right)$, we can solve $\hat{H}\left(X\right)$ where $\hat{H}\left(X\right): \mathbb{R}^{T}\rightarrow \mathbb{R}^T$ is defined as

\begin{align}
\hat{H}\left(X\right)=
\begin{bmatrix}
h\left(x_{-1},x_0,x_1\right) \\
    h\left(x_0,x_1,x_2\right) \\
    \vdots \\
    h\left(x_{T-1},x_{T},x_{T+1}\right)
\end{bmatrix}
\end{align}

Now we have a finite dimensional system of equations that we could in principal use a numerical root finder to solve. However, this may not be the best method. Let's first note that $h\left(x_{j-1},x_j,x_{j+1}\right)=0$ if and only if the following is true
\begin{align}
    x_{j+1}=x_{t}^{\alpha}+\left(1-\delta\right)x_{t}-\left(x_{t-1}^{\alpha}+\left(1-\delta\right)x_{t-1}-x_t\right)\left[\beta \left(\alpha x_t^{\alpha-1}+\left(1-\delta\right)\right)\right]^{\frac{1}{\gamma}}
\end{align}

What this says is that if I know $\left\{x_{j-1},x_j\right\}$, then I know $x_{j+1}$. Another way to look at the expression is as a second order difference equation in $x$. As you may recall from a differential equations course, we need two boundary conditions for a solution to a second order differential equation. The same is true for a second order difference equation. Here, our boundary conditions are $k_{-1}=\tilde{k}$ (given) and $\lim_{t\rightarrow \infty} k_t=k^*$. Just as you could do with differential equations, we can use a forward shooting method to solve our problem. 

## Forward Shooting Method
From the equation 

\begin{align}
    x_{j+1}=x_{t}^{\alpha}+\left(1-\delta\right)x_{t}-\left(x_{t-1}^{\alpha}+\left(1-\delta\right)x_{t-1}-x_t\right)\left[\beta \left(\alpha x_t^{\alpha-1}+\left(1-\delta\right)\right)\right]^{\frac{1}{\gamma}}
\end{align}

we see that if we were given $x_0$, then we would be able to move the equation forward in time and generate an entire sequence of $x_t$'s, $\left\{x_t\right\}_{t=0}^{T}$. Our boundary condition, $\lim_{t\rightarrow \infty} k_t=k^*$, tells us that for $T$ large enough $\left|x_{T-1}-x_{T}\right|<\epsilon$ (a convergent sequence is Cauchy).

These two observations suggest the following algorithm.

1. Guess $k_0$.
2. Iterate forward for $T$ periods.
3. Compute $\mathtt{dif}=|k_{T-1}-k_T|$ and $\mathtt{sign}=sign\left(k_{T-1}-k_T\right)$
4. If $\mathtt{dif}<\mathtt{tol}$, stop. Else, return to 1.

The only thing we're missing is a method for updating our guess for $k_0$. Consider the case when $\mathtt{sign}>0$. If $\mathtt{sign}>0$, this means that we overshot the steady state. What does this mean about our choice of $k_0$? If we overshot the steady state, then that means we saved too much initially. Therefore, we should revise our initial guess downward.

What should we do if $\mathtt{sign}<0$. This means that $k_{T-1}<k_T$ so we undershot the steady state. Therefore, we didn't save enough in the initial period. Therefore, if $\mathtt{sign}<0$ we should revise $k_0$ upwards.

The previous observations suggest that we should use bisection to update our guess. 
0. Initialize $k_L$ and $k_H$
1. Compute $k_0:=\frac{k_L+k_H}{2}$
2. Compute the path for variables as described above
3. If $\mathtt{sign}>0$, set $k_L:=k_0$. Else set $k_H:=k_0$. Return to step 1.

Now, let's turn to the Matlab code to see how to implement this.

In [ ]:
clc;
clear all;
tol=0.0000001; %Maximum tolerance of Norm(Vi-Vi-1) for convergence.
maxiter=10000; %Maximum number of iterations.
dif=10; %initialize norm of the diference of Vi an Vi-1
iter=1; %initialize number of iterations
alpha=0.3;
beta=0.99;
delta=.1;

% model parameters
mp=struct("alpha",alpha,"beta",beta,"delta",delta)

[kstar,cstar]=compute_steady_state(mp);

k00=.5;
c_L=0;
c_H=k00^alpha+(1-delta)*k00;
MaxT=200;
tic;
while dif>tol & iter<maxiter
     k0=0.00001;
     c0=(c_L+c_H)/2;
     [dif,sign,C0,K0,K1]=forward_pass(c0,k0,mp,MaxT,dif,tol);
     [c_L,c_H]=update_bounds(sign,c_L,c_H);
     [MaxT,dif]=update_MaxT(K1(end,1),kstar,MaxT,tol,dif);
     iter=iter+1;
end
toc;
T=size(C0,1);
time=[1:1:T];
CS=ones(T,1)*cstar;
KS=ones(T,1)*kstar;
figure()
subplot(1,2,1)
plot(time,C0,time,CS)
title('Consumption')
subplot(1,2,2)
plot(time,K1,time,KS)
title('Capital')

The code calls four functions. As you start becoming more comfortable with Matlab (or any programming language), you should try to use more, small functions rather than writing large chunks of code in the body of the main code.

The first function, $\mathtt{compute}\_\mathtt{steady}\_\mathtt{state}\left(mp\right)$, takes in the parameters of the model and returns the steady state, $\left(cstar,kstar\right)$


In [ ]:
function [kstar,cstar]=compute_steady_state(mp)
    kstar=(mp.alpha/((1.0/mp.beta)-(1.0-mp.delta)))^(1.0/(1.0-mp.alpha));
    cstar=kstar^mp.alpha-mp.delta*kstar;
end

The three functions used in each iteration are

* $\mathtt{forward}\_\mathtt{pass}\left(c0,k0,mp,MaxT\right)$
* $\mathtt{update}\_\mathtt{bounds}\left(sign,c_L,c_H\right)$
* $\mathtt{update}\_\mathtt{MaxT}\left(kend,kstar,MaxT\right)$

$\mathtt{forward}\_\mathtt{pass}\left(c0,k0,mp,MaxT\right)$ takes in $c0$ (our guess), $k0$ (our initial capital), mp (the parameters of the model) and $MaxT$ (the maximum number of periods we want to simulate the model forward for).

In [ ]:
function [dif,sign,C0,K0,K1]=forward_pass(c0,k0,mp,MaxT,dif,tol)
    C0=zeros(MaxT+1,1);
    K0=zeros(MaxT+1,1);
    K1=zeros(MaxT+1,1);
    T=1;
    while (T<=MaxT) & (dif>tol) & (k0>0)
        % Capital Accumulation Equation
        k1=k0^mp.alpha+(1.0-mp.delta)*k0-c0;
        % Euler Equation
        c1=c0*(mp.beta*(mp.alpha*k1^(mp.alpha-1.0)+(1.0-mp.delta)));
        C0(T,1)=c0;
        K0(T,1)=k0;
        K1(T,1)=k1;
        dif=abs((k1-k0)/k0);
        sign=k1-k0;
        c0=c1;
        k0=k1;
        T=T+1;
    end  
    C0=C0(1:T-1,1);
    K0=K0(1:T-1,1);
    K1=K1(1:T-1,1);
end

$\mathtt{update}\_\mathtt{bounds}\left(sign,c_L,c_H\right)$ uses the sign returned from$\mathtt{forward}\_\mathtt{pass}$ to update the bounds for $c0$.

In [ ]:
function [c_L,c_H]=update_bounds(sign,c_L,c_H)
    if sign > 0
         % Sign>0 => didn't consume enough in period 0 so we adjust upward
         % our starting consumption level
         c_L=(c_L+c_H)/2;
    else
         % k1-k0<0 => we consumed too much in period 0 so we adjust
         % downward our starting consumption level
         c_H=(c_L+c_H)/2;
     end
end

Finally, $\mathtt{update}\_\mathtt{MaxT}\left(kend,kstar,MaxT\right)$ uses the last value of $k$ computed in the forward pass to to see if we should increase or decrease $MaxT$. To see why we need to do this, suppose we set $MaxT=20$ and $k0=\frac{1}{1000}k^*$. Clearly, 20 periods is not enough time to get from such a low value of initial capital to steady state. Therefore, we won't be able to find a guess of $c0$ that gets us to steady state. By allowing $MaxT$ to adjust, we can avoid the non-convergence issues due to not using a long enough forward simulation.

In [ ]:
function [MaxT,dif]=update_MaxT(kend,kstar,MaxT,tol,dif)
     if (kend-kstar)>tol
         MaxT=MaxT-10;
         dif=10;
     elseif (kstar-kend)>tol
         MaxT=MaxT+10;
         dif=dif;
     else
         MaxT=MaxT;
     end
end